# Notebook 3c — Baryon Acoustic Oscillations

The BAO scale imprints a characteristic length on the galaxy distribution — the **sound horizon** at the drag epoch $r_d \approx 147$ Mpc. By measuring where this scale appears in the sky (transverse) and along the line of sight (radial), we can constrain both distances and the expansion rate.

The three observables are:

$$\frac{D_M(z)}{r_d} \qquad \frac{D_H(z)}{r_d} \qquad \frac{D_V(z)}{r_d}$$

where $D_M = \chi(z)$ is the comoving distance, $D_H = c/H(z)$ is the Hubble distance, and $D_V = [z\,D_M^2\,D_H]^{1/3}$ is the angle-averaged distance.

We use **DESI DR2** data (Abdul Karim et al. 2025, arXiv:2503.14738).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from plots import *
from chi2 import chi2_bao

---
## 1. Load the data `[S1]`

In [ ]:
# columns: tracer, z_eff, observable, value, sigma
data = np.genfromtxt('../data/desi_dr2_bao.txt', dtype=None,
                     encoding=None, names=['tracer','z','obs','val','err'])

# split by observable type
mask_DM = data['obs'] == 'DM_rd'
mask_DH = data['obs'] == 'DH_rd'
mask_DV = data['obs'] == 'DV_rd'

z_DM,  DM_obs,  dDM  = data['z'][mask_DM], data['val'][mask_DM], data['err'][mask_DM]
z_DH,  DH_obs,  dDH  = data['z'][mask_DH], data['val'][mask_DH], data['err'][mask_DH]
z_DV,  DV_obs,  dDV  = data['z'][mask_DV], data['val'][mask_DV], data['err'][mask_DV]

print(f"DM/rd measurements: {len(z_DM)}  at z = {z_DM}")
print(f"DH/rd measurements: {len(z_DH)}  at z = {z_DH}")
print(f"DV/rd measurements: {len(z_DV)}  at z = {z_DV}")

---
## 2. The sound horizon `[S2]`

The sound horizon at the drag epoch is computed from a fitting formula (Eisenstein & Hu 1998):

$$r_d = 147.05\left(\frac{\Omega_m h^2}{0.1432}\right)^{-0.32} \text{ Mpc}$$

This depends on both $\Omega_m$ and $h$, so BAO constrains them jointly.

In [ ]:
def rd(Om, h):
    """Sound horizon at the drag epoch in Mpc (fitting formula)."""
    return 147.05 * (Om * h**2 / 0.1432)**(-0.32)

# fiducial values
print(f"rd(Om=0.30, h=0.70) = {rd(0.30, 0.70):.2f} Mpc")
print(f"rd(Om=0.31, h=0.68) = {rd(0.31, 0.68):.2f} Mpc")

---
## 3. Theory predictions `[S3]`

From `solve_distances` we get $d_c(z)$ (comoving distance in units of $c/H_0$). Converting:

$$D_M(z) = d_c(z) \cdot D_H(h) \quad [\text{Mpc}]$$
$$D_H(z) = \frac{1}{E(z)} \cdot D_H(h) \quad [\text{Mpc}]$$
$$D_V(z) = \left[z\,D_M^2(z)\,D_H(z)\right]^{1/3} \quad [\text{Mpc}]$$

In [ ]:
Om = 0.30
h  = 0.70
rho_de, _ = de_model('LCDM', Om)

z_max_plot = 2.5
z_th, d_c_th, _ = solve_distances(Om, rho_de, z_max=z_max_plot, n_points=500)

# convert to Mpc
DM_th = d_c_th * DH(h)                                    # comoving distance [Mpc]
DH_th = DH(h) / np.sqrt(E2(z_to_a(z_th), Om, rho_de))    # Hubble distance [Mpc]
DV_th = (z_th * DM_th**2 * DH_th)**(1/3)                  # volume distance [Mpc]

# divide by rd
rd_fid = rd(Om, h)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].plot(z_th[1:], DM_th[1:]/rd_fid, color='#2a78d6', lw=2)
axes[0].errorbar(z_DM, DM_obs, yerr=dDM, fmt='o', color='#e34948', ms=6, capsize=3)
axes[0].set_xlabel(r'$z$')
axes[0].set_ylabel(r'$D_M/r_d$')
axes[0].set_title(r'Comoving distance')

axes[1].plot(z_th[1:], DH_th[1:]/rd_fid, color='#2a78d6', lw=2)
axes[1].errorbar(z_DH, DH_obs, yerr=dDH, fmt='o', color='#e34948', ms=6, capsize=3)
axes[1].set_xlabel(r'$z$')
axes[1].set_ylabel(r'$D_H/r_d$')
axes[1].set_title(r'Hubble distance')

axes[2].plot(z_th[1:], DV_th[1:]/rd_fid, color='#2a78d6', lw=2)
axes[2].errorbar(z_DV, DV_obs, yerr=dDV, fmt='o', color='#e34948', ms=6, capsize=3)
axes[2].set_xlabel(r'$z$')
axes[2].set_ylabel(r'$D_V/r_d$')
axes[2].set_title(r'Volume distance')

for ax in axes:
    ax.set_xlim(0, z_max_plot)

plt.suptitle(f'DESI DR2 BAO — ΛCDM ($\\Omega_m={Om:g}$, $h={h:g}$)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. $\chi^2$ scan over $(\Omega_m, h)$ `[S4]`

In [ ]:
Om_arr = np.linspace(0.25, 0.35, 60)
h_arr  = np.linspace(0.65, 0.75, 60)
chi2_grid = np.zeros((len(Om_arr), len(h_arr)))

for i, Om in enumerate(Om_arr):
    rho_de, _ = de_model('LCDM', Om)
    # compute distances once per Om
    z_max = max(z_DM.max(), z_DH.max(), z_DV.max()) * 1.01
    z_th, d_c_th, _ = solve_distances(Om, rho_de, z_max=z_max, n_points=1000)
    for j, h in enumerate(h_arr):
        chi2_grid[i, j] = chi2_bao(Om, h, rho_de,
                                    z_th, d_c_th,
                                    z_DM, DM_obs, dDM,
                                    z_DH, DH_obs, dDH,
                                    z_DV, DV_obs, dDV)

chi2_min = chi2_grid.min()
idx = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)
Om_best = Om_arr[idx[0]]
h_best  = h_arr[idx[1]]

print(f"Best fit Om = {Om_best:g}")
print(f"Best fit h  = {h_best:g}")
print(f"chi2_min    = {chi2_min:g}")
print(f"chi2/dof    = {chi2_min/(len(z_DM)+len(z_DH)+len(z_DV)-2):g}")

fig, ax = plt.subplots(figsize=(7, 6))
ct = ax.contour(Om_arr, h_arr, (chi2_grid - chi2_min).T,
                levels=[2.30, 6.17], colors=['#2a78d6', '#e34948'])
ax.clabel(ct, fmt={2.30: r'$1\sigma$', 6.17: r'$2\sigma$'}, fontsize=10)
ax.plot(Om_best, h_best, 'k*', ms=10, label=f'best fit')
ax.set_xlabel(r'$\Omega_m$')
ax.set_ylabel(r'$h$')
ax.set_title(r'DESI DR2 BAO — $\chi^2(\Omega_m, h)$')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 5. 1D scans `[S5]`

In [ ]:
# scan over h, fix Om at best fit
rho_de, _ = de_model('LCDM', Om_best)
z_max = max(z_DM.max(), z_DH.max(), z_DV.max()) * 1.01
z_th, d_c_th, _ = solve_distances(Om_best, rho_de, z_max=z_max, n_points=1000)
chi2_h = np.array([chi2_bao(Om_best, h, rho_de, z_th, d_c_th,
                             z_DM, DM_obs, dDM,
                             z_DH, DH_obs, dDH,
                             z_DV, DV_obs, dDV) for h in h_arr])

# scan over Om, fix h at best fit
chi2_Om = np.zeros(len(Om_arr))
for i, Om in enumerate(Om_arr):
    rho_de, _ = de_model('LCDM', Om)
    z_th, d_c_th, _ = solve_distances(Om, rho_de, z_max=z_max, n_points=1000)
    chi2_Om[i] = chi2_bao(Om, h_best, rho_de, z_th, d_c_th,
                           z_DM, DM_obs, dDM,
                           z_DH, DH_obs, dDH,
                           z_DV, DV_obs, dDV)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(h_arr, chi2_h - chi2_h.min(), color='#2a78d6')
axes[0].axvline(h_best, color='#e34948', ls='--', lw=1.5, label=f'$h = {h_best:g}$')
axes[0].axhline(1, color='black', ls=':', lw=1, alpha=0.5)
axes[0].set_xlabel(r'$h$')
axes[0].set_ylabel(r'$\Delta\chi^2$')
axes[0].set_title(r'Fixed $\Omega_m$')
axes[0].legend(fontsize=10)

axes[1].plot(Om_arr, chi2_Om - chi2_Om.min(), color='#2a78d6')
axes[1].axvline(Om_best, color='#e34948', ls='--', lw=1.5,
                label=f'$\\Omega_m = {Om_best:g}$')
axes[1].axhline(1, color='black', ls=':', lw=1, alpha=0.5)
axes[1].set_xlabel(r'$\Omega_m$')
axes[1].set_ylabel(r'$\Delta\chi^2$')
axes[1].set_title(r'Fixed $h$')
axes[1].legend(fontsize=10)

plt.suptitle('DESI DR2 BAO — 1D scans')
plt.tight_layout()
plt.show()